# 🔫 Weapon Detection — Research Paper Implementation
## Custom Multi-Source Dataset Training with YOLOv8 Ablation Study & 5-Fold Cross-Validation

---

**Paper Implementation Summary:**
- **Dataset:** Custom multi-source (Roboflow Universe, Google Open Images v7, Kaggle, Synthetic Augmentation)
- **Classes:** Handgun (0), Knife (1), Rifle (2), Shotgun (3)
- **Models:** YOLOv8n, YOLOv8s, YOLOv8m (ablation study)
- **Validation:** 5-Fold Cross-Validation (80/20 per fold), max 100 epochs, early stopping patience=15
- **Optimizer:** SGD, momentum=0.937, weight_decay=5e-4, cosine annealing LR
- **Inference:** NMS IoU=0.45, confidence=0.25

> ⚠️ **Runtime:** Set Runtime → Change Runtime Type → **GPU (T4 or A100)**

---
**Estimated Time:** ~6-10 hours for full pipeline (depending on GPU tier)

## ⚙️ QUICK START — Configure All Variables Here
**Edit this cell before running anything else.**

In [ ]:
# ==============================================================
# QUICK START CONFIGURATION — Edit these before running
# ==============================================================

# --- API Keys ---
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"   # Get free key at https://app.roboflow.com
KAGGLE_USERNAME  = "YOUR_KAGGLE_USERNAME"    # Your Kaggle username
KAGGLE_KEY       = "YOUR_KAGGLE_KEY"         # Get from kaggle.com → Account → API token

# --- Google Drive ---
DRIVE_BASE = "/content/drive/MyDrive/WeaponDetection"  # All outputs saved here

# --- Dataset Settings ---
OPEN_IMAGES_MAX_SAMPLES = 3000   # Max images from Open Images per split
NUM_FOLDS               = 5      # Number of cross-validation folds
TRAIN_RATIO             = 0.70   # 70% train
VAL_RATIO               = 0.15   # 15% val
TEST_RATIO              = 0.15   # 15% test

# --- Training Hyperparameters (Paper-Exact) ---
EPOCHS          = 100
PATIENCE        = 15      # Early stopping
BATCH           = 16
IMGSZ           = 640
LR0             = 0.01    # Initial LR
LRF             = 0.001   # Final LR (cosine annealing target)
MOMENTUM        = 0.937
WEIGHT_DECAY    = 5e-4
OPTIMIZER       = 'SGD'
WARMUP_EPOCHS   = 3

# --- Augmentation (Paper-Exact) ---
AUG_MOSAIC      = 0.9
AUG_FLIPLR      = 0.5
AUG_SCALE       = 0.5
AUG_TRANSLATE   = 0.1
AUG_HSV_H       = 0.015
AUG_HSV_S       = 0.7
AUG_HSV_V       = 0.4
AUG_ERASING     = 0.3
AUG_MIXUP       = 0.15

# --- Inference ---
NMS_IOU_THRESH  = 0.45
CONF_THRESH     = 0.25

# --- Classes ---
CLASS_NAMES = ['Handgun', 'Knife', 'Rifle', 'Shotgun']

# --- Models for Ablation ---
MODELS_TO_TRAIN = [
    ('yolov8n', 'yolov8n.pt'),
    ('yolov8s', 'yolov8s.pt'),
    ('yolov8m', 'yolov8m.pt'),
]
BEST_MODEL_NAME = 'yolov8s'

print("✅ Configuration loaded.")
print(f"   Drive base: {DRIVE_BASE}")
print(f"   Classes: {CLASS_NAMES}")
print(f"   Models: {[m[0] for m in MODELS_TO_TRAIN]}")
print(f"   Epochs: {EPOCHS}, Patience: {PATIENCE}, Batch: {BATCH}, ImgSz: {IMGSZ}")

---
# Section 1: Environment Setup
Install all required packages, verify GPU, and mount Google Drive.

In [ ]:
# 1.1 — Install Required Packages
print("Installing packages...")
!pip install -q ultralytics>=8.0.0
!pip install -q roboflow
!pip install -q fiftyone
!pip install -q kaggle
!pip install -q open-images-downloader
!pip install -q tqdm matplotlib seaborn pandas scikit-learn Pillow opencv-python-headless
print("✅ All packages installed.")

In [ ]:
# 1.2 — Verify GPU Availability
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU detected: {gpu_name} ({gpu_mem:.1f} GB)")
    if 'T4' in gpu_name:
        print("   → NVIDIA T4 — Good for training (15 GB VRAM)")
    elif 'A100' in gpu_name:
        print("   → NVIDIA A100 — Excellent for training (40/80 GB VRAM)")
    else:
        print(f"   → {gpu_name} — Training will proceed")
else:
    print("⚠️  No GPU detected! Go to Runtime → Change Runtime Type → GPU")
    print("   Training will be very slow on CPU.")

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"   Device set to: {DEVICE}")

# Check ultralytics
import ultralytics
print(f"   Ultralytics version: {ultralytics.__version__}")

In [ ]:
# 1.3 — Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Create project directory structure on Drive
dirs_to_create = [
    DRIVE_BASE,
    f"{DRIVE_BASE}/dataset",
    f"{DRIVE_BASE}/dataset/raw",
    f"{DRIVE_BASE}/dataset/raw/roboflow",
    f"{DRIVE_BASE}/dataset/raw/openimages",
    f"{DRIVE_BASE}/dataset/raw/kaggle",
    f"{DRIVE_BASE}/dataset/raw/synthetic",
    f"{DRIVE_BASE}/dataset/unified",
    f"{DRIVE_BASE}/dataset/unified/images",
    f"{DRIVE_BASE}/dataset/unified/labels",
    f"{DRIVE_BASE}/dataset/test",
    f"{DRIVE_BASE}/dataset/test/images",
    f"{DRIVE_BASE}/dataset/test/labels",
    f"{DRIVE_BASE}/weights",
    f"{DRIVE_BASE}/results",
    f"{DRIVE_BASE}/logs",
]
for d in dirs_to_create:
    os.makedirs(d, exist_ok=True)

print(f"✅ Google Drive mounted and directory structure created under:")
print(f"   {DRIVE_BASE}")

In [ ]:
# 1.4 — Global Imports & Utility Setup
import os, sys, json, shutil, glob, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
from pathlib import Path
from tqdm import tqdm
from PIL import Image, ImageFilter
from sklearn.model_selection import StratifiedKFold
from collections import Counter, defaultdict
from ultralytics import YOLO
import yaml

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# Local working directory (fast SSD on Colab)
LOCAL_BASE   = '/content/weapon_detection'
LOCAL_DATA   = f'{LOCAL_BASE}/dataset'
LOCAL_FOLDS  = f'{LOCAL_BASE}/folds'
LOCAL_RUNS   = f'{LOCAL_BASE}/runs'

for d in [LOCAL_BASE, LOCAL_DATA, LOCAL_FOLDS, LOCAL_RUNS]:
    os.makedirs(d, exist_ok=True)

print("✅ Imports complete. Local workspace:", LOCAL_BASE)

---
# Section 2: Dataset Assembly (Custom Multi-Source)
**Per the paper:** Dataset assembled from Roboflow Universe, Google Open Images v7, Kaggle weapon repositories, and synthetic augmentation for underrepresented conditions.

Each source is downloaded independently. Failures are handled gracefully — if one source fails, the pipeline continues with remaining sources.

In [ ]:
# 2.0 — Dataset Source Tracker
# Tracks which sources were successfully downloaded
SOURCE_STATUS = {
    'roboflow':    {'success': False, 'images': 0, 'path': None},
    'openimages':  {'success': False, 'images': 0, 'path': None},
    'kaggle':      {'success': False, 'images': 0, 'path': None},
    'synthetic':   {'success': False, 'images': 0, 'path': None},
}

def log_source(name, success, images=0, path=None):
    SOURCE_STATUS[name]['success'] = success
    SOURCE_STATUS[name]['images']  = images
    SOURCE_STATUS[name]['path']    = path
    status = '✅' if success else '⚠️ SKIPPED'
    print(f"  [{status}] {name}: {images} images")

print("Dataset source tracker initialized.")

### Source A — Roboflow Universe Weapon Datasets
Roboflow Universe hosts hundreds of public weapon detection datasets. We download multiple projects and merge them. **Get your free API key at [app.roboflow.com](https://app.roboflow.com)**.

In [ ]:
# 2A — Roboflow Universe Weapon Datasets
# Multiple public weapon detection projects from Roboflow Universe

ROBOFLOW_PROJECTS = [
    # Format: (workspace_id, project_id, version)
    # These are known public weapon detection projects on Roboflow Universe
    ('roboflow-100',   'weapons-detection-jxtff',    3),
    ('roboflow-100',   'pistol-detection-ozkuk',     1),
    ('joseph-nelson',  'open-images-weapon-dataset', 1),
]

RF_OUTPUT = f"{LOCAL_DATA}/raw/roboflow"
os.makedirs(RF_OUTPUT, exist_ok=True)

roboflow_images = 0

try:
    if ROBOFLOW_API_KEY == 'YOUR_ROBOFLOW_API_KEY':
        raise ValueError("Roboflow API key not set. Update ROBOFLOW_API_KEY in Quick Start cell.")

    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)

    for ws_id, proj_id, version in ROBOFLOW_PROJECTS:
        try:
            print(f"  Downloading {ws_id}/{proj_id} v{version}...")
            project  = rf.workspace(ws_id).project(proj_id)
            dataset  = project.version(version).download(
                'yolov8',
                location=f"{RF_OUTPUT}/{proj_id}",
                overwrite=True
            )
            # Count downloaded images
            imgs = glob.glob(f"{RF_OUTPUT}/{proj_id}/**/*.jpg", recursive=True)
            imgs += glob.glob(f"{RF_OUTPUT}/{proj_id}/**/*.png", recursive=True)
            roboflow_images += len(imgs)
            print(f"    → {len(imgs)} images downloaded from {proj_id}")
        except Exception as e:
            print(f"    ⚠️  Skipped {proj_id}: {e}")
            continue

    log_source('roboflow', True, roboflow_images, RF_OUTPUT)

except Exception as e:
    print(f"\n⚠️  Roboflow download failed: {e}")
    print("   Continuing with other sources...")
    print("\n📋 MANUAL INSTRUCTIONS for Roboflow:")
    print("   1. Go to https://universe.roboflow.com")
    print("   2. Search for 'weapon detection'")
    print("   3. Download any public dataset in YOLOv8 format")
    print(f"   4. Upload extracted folder to: {RF_OUTPUT}")
    log_source('roboflow', False, 0)

### Source B — Google Open Images v7 (via FiftyOne)
Open Images v7 contains annotated 'Handgun', 'Knife', and 'Rifle' classes. We download up to 3000 samples per split and export in YOLO format.

> **Note:** Shotgun is not a separate class in Open Images; those images will come from Roboflow/Kaggle.

In [ ]:
# 2B — Google Open Images v7 via FiftyOne
import fiftyone as fo
import fiftyone.zoo as foz

OI_OUTPUT = f"{LOCAL_DATA}/raw/openimages"
OI_YOLO   = f"{OI_OUTPUT}/yolo_format"
os.makedirs(OI_OUTPUT, exist_ok=True)
os.makedirs(OI_YOLO, exist_ok=True)

# Classes available in Open Images (Shotgun not present, will use Roboflow/Kaggle)
OI_CLASSES = ['Handgun', 'Knife', 'Rifle']

oi_images = 0

try:
    print(f"Downloading Open Images v7 classes: {OI_CLASSES}")
    print(f"Max samples: {OPEN_IMAGES_MAX_SAMPLES} per split")
    print("This may take 15-30 minutes...")

    for split in ['train', 'validation']:
        print(f"\n  → Downloading split: {split}")
        try:
            dataset = foz.load_zoo_dataset(
                'open-images-v7',
                split=split,
                label_types=['detections'],
                classes=OI_CLASSES,
                max_samples=OPEN_IMAGES_MAX_SAMPLES,
                dataset_name=f'oi_weapons_{split}',
            )
            print(f"    Loaded {len(dataset)} samples from Open Images {split}")

            # Export in YOLO format
            split_dir = f"{OI_YOLO}/{split}"
            dataset.export(
                export_dir=split_dir,
                dataset_type=fo.types.YOLOv5Dataset,
                label_field='ground_truth',
                classes=OI_CLASSES,
            )

            imgs = glob.glob(f"{split_dir}/**/*.jpg", recursive=True)
            imgs += glob.glob(f"{split_dir}/**/*.png", recursive=True)
            oi_images += len(imgs)
            print(f"    Exported {len(imgs)} images to {split_dir}")

            # Cleanup FiftyOne dataset to save disk space
            dataset.delete()

        except Exception as e:
            print(f"    ⚠️  Error on split {split}: {e}")
            continue

    log_source('openimages', oi_images > 0, oi_images, OI_YOLO)

except Exception as e:
    print(f"\n⚠️  Open Images download failed: {e}")
    print("   Continuing with other sources...")
    log_source('openimages', False, 0)

### Source C — Kaggle Weapon Detection Datasets
Upload your `kaggle.json` credentials file (from kaggle.com → Account → Create API Token) and download weapon detection datasets.

In [ ]:
# 2C.1 — Kaggle API Setup
# Option 1: Upload kaggle.json via Colab file upload
# Option 2: Set credentials via environment variables (used if API key provided above)

KAGGLE_OUTPUT = f"{LOCAL_DATA}/raw/kaggle"
os.makedirs(KAGGLE_OUTPUT, exist_ok=True)
kaggle_images = 0

# Set up Kaggle credentials
if KAGGLE_USERNAME != 'YOUR_KAGGLE_USERNAME' and KAGGLE_KEY != 'YOUR_KAGGLE_KEY':
    # Use credentials from Quick Start
    os.makedirs('/root/.kaggle', exist_ok=True)
    kaggle_creds = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump(kaggle_creds, f)
    !chmod 600 /root/.kaggle/kaggle.json
    print("✅ Kaggle credentials set from Quick Start config.")
else:
    print("📋 Kaggle credentials not configured. Choose an option:")
    print("   OPTION A: Update KAGGLE_USERNAME and KAGGLE_KEY in Quick Start cell")
    print("   OPTION B: Upload kaggle.json manually (run cell below)")
    print("   OPTION C: Skip Kaggle — other sources may be sufficient")

In [ ]:
# 2C.2 — (OPTIONAL) Upload kaggle.json Manually
# Run this cell only if you haven't set credentials in Quick Start

try:
    from google.colab import files
    print("Click 'Choose Files' to upload your kaggle.json")
    uploaded = files.upload()

    if 'kaggle.json' in uploaded:
        os.makedirs('/root/.kaggle', exist_ok=True)
        with open('/root/.kaggle/kaggle.json', 'wb') as f:
            f.write(uploaded['kaggle.json'])
        !chmod 600 /root/.kaggle/kaggle.json
        print("✅ kaggle.json uploaded and configured.")
    else:
        print("⚠️  kaggle.json not found in upload. Skipping Kaggle.")
except Exception as e:
    print(f"⚠️  Upload skipped: {e}")

In [ ]:
# 2C.3 — Download Kaggle Weapon Detection Datasets
# Known public weapon detection datasets on Kaggle

KAGGLE_DATASETS = [
    # (dataset_slug, description)
    ('sayansadhukhan/weapons-detection-yolo-format',        'Weapons Detection YOLO Format'),
    ('intelecai/gun-detection',                             'Gun Detection Dataset'),
    ('dataturks/dataturks-weapon-detection-dataset',        'DataTurks Weapon Detection'),
]

try:
    import kaggle
    kaggle.api.authenticate()
    print("✅ Kaggle API authenticated.")

    for dataset_slug, desc in KAGGLE_DATASETS:
        try:
            safe_name = dataset_slug.split('/')[-1]
            dest_dir  = f"{KAGGLE_OUTPUT}/{safe_name}"
            os.makedirs(dest_dir, exist_ok=True)

            print(f"  Downloading: {desc} ({dataset_slug})")
            !kaggle datasets download -d {dataset_slug} -p {dest_dir} --unzip -q

            imgs = glob.glob(f"{dest_dir}/**/*.jpg", recursive=True)
            imgs += glob.glob(f"{dest_dir}/**/*.png", recursive=True)
            kaggle_images += len(imgs)
            print(f"    → {len(imgs)} images downloaded")

        except Exception as e:
            print(f"    ⚠️  Skipped {dataset_slug}: {e}")
            continue

    log_source('kaggle', kaggle_images > 0, kaggle_images, KAGGLE_OUTPUT)

except Exception as e:
    print(f"\n⚠️  Kaggle download failed: {e}")
    print("   Continuing without Kaggle data...")
    log_source('kaggle', False, 0)

### Source D — Synthetic Augmentation for Adverse Conditions
**Per the paper:** Synthetic augmentation is applied to minority-class images to simulate:
1. **Low illumination** — Gamma darkening (γ ∈ [0.3, 0.6]) + Poisson noise
2. **Motion blur** — Directional Gaussian blur, kernel 7–15px, random angle
3. **Occlusion** — Random rectangular masking patches

These synthetic variants are added to the training pool to improve robustness.

In [ ]:
# 2D.1 — Synthetic Augmentation Functions

def apply_gamma_darkening(image_bgr, gamma=None):
    """Low-light simulation: gamma darkening (gamma in [0.3, 0.6]) + Poisson noise."""
    if gamma is None:
        gamma = random.uniform(0.3, 0.6)

    # Gamma correction
    inv_gamma = 1.0 / gamma
    table = np.array([
        ((i / 255.0) ** inv_gamma) * 255
        for i in range(256)
    ]).astype('uint8')
    dark_img = cv2.LUT(image_bgr, table)

    # Poisson noise
    vals  = len(np.unique(dark_img))
    vals  = 2 ** np.ceil(np.log2(vals))
    noise = np.random.poisson(dark_img / 255.0 * vals) / float(vals) * 255
    noisy = np.clip(noise, 0, 255).astype(np.uint8)
    return noisy


def apply_motion_blur(image_bgr, kernel_size=None, angle=None):
    """Motion blur: directional Gaussian kernel, kernel size 7-15px, random angle."""
    if kernel_size is None:
        kernel_size = random.choice(range(7, 16, 2))  # 7, 9, 11, 13, 15
    if angle is None:
        angle = random.uniform(0, 180)

    # Create motion blur kernel
    kernel = np.zeros((kernel_size, kernel_size))
    kernel[kernel_size // 2, :] = 1.0
    kernel = kernel / kernel_size

    # Rotate kernel
    M = cv2.getRotationMatrix2D(
        (kernel_size / 2, kernel_size / 2), angle, 1
    )
    kernel = cv2.warpAffine(kernel, M, (kernel_size, kernel_size))
    if kernel.sum() > 0:
        kernel /= kernel.sum()

    blurred = cv2.filter2D(image_bgr, -1, kernel)
    return blurred


def apply_occlusion(image_bgr, num_patches=None, patch_ratio=0.15):
    """Partial occlusion: random rectangular masking patches."""
    if num_patches is None:
        num_patches = random.randint(1, 4)

    img   = image_bgr.copy()
    h, w  = img.shape[:2]

    for _ in range(num_patches):
        ph = int(h * patch_ratio * random.uniform(0.5, 1.5))
        pw = int(w * patch_ratio * random.uniform(0.5, 1.5))
        ph = min(ph, h - 1)
        pw = min(pw, w - 1)
        x1 = random.randint(0, w - pw)
        y1 = random.randint(0, h - ph)
        # Fill with random color or black
        color = [random.randint(0, 50)] * 3  # Dark patch
        img[y1:y1 + ph, x1:x1 + pw] = color

    return img


SYNTHETIC_TRANSFORMS = {
    'lowlight':  apply_gamma_darkening,
    'motionblur': apply_motion_blur,
    'occlusion': apply_occlusion,
}

print("✅ Synthetic augmentation functions defined:")
for t in SYNTHETIC_TRANSFORMS:
    print(f"   → {t}")

In [ ]:
# 2D.2 — Generate Synthetic Augmented Images
# Applied to minority-class images from existing sources

SYNTH_OUTPUT = f"{LOCAL_DATA}/raw/synthetic"
os.makedirs(f"{SYNTH_OUTPUT}/images", exist_ok=True)
os.makedirs(f"{SYNTH_OUTPUT}/labels", exist_ok=True)

def generate_synthetic_from_source(source_img_dir, source_lbl_dir, output_dir, max_per_transform=300):
    """Generate synthetic augmented copies of images with their original labels."""
    img_files = glob.glob(f"{source_img_dir}/**/*.jpg", recursive=True)
    img_files += glob.glob(f"{source_img_dir}/**/*.png", recursive=True)

    if not img_files:
        print(f"  No images found in {source_img_dir}")
        return 0

    # Sample randomly
    sample = random.sample(img_files, min(max_per_transform, len(img_files)))
    synth_count = 0

    for transform_name, transform_fn in SYNTHETIC_TRANSFORMS.items():
        print(f"  Applying {transform_name} to {len(sample)} images...")
        for img_path in tqdm(sample, desc=f"  {transform_name}", leave=False):
            try:
                img = cv2.imread(img_path)
                if img is None:
                    continue

                aug_img = transform_fn(img)
                stem    = Path(img_path).stem
                out_img = f"{output_dir}/images/{stem}_{transform_name}.jpg"
                cv2.imwrite(out_img, aug_img)

                # Copy corresponding label
                lbl_path = Path(source_lbl_dir) / (stem + '.txt')
                if lbl_path.exists():
                    out_lbl = f"{output_dir}/labels/{stem}_{transform_name}.txt"
                    shutil.copy(str(lbl_path), out_lbl)

                synth_count += 1
            except Exception:
                continue

    return synth_count


# Find available source images for synthetic generation
synth_total = 0
print("Generating synthetic augmented images...")

# Try each source
for src_name, src_info in SOURCE_STATUS.items():
    if src_info['success'] and src_info['path']:
        img_dirs = glob.glob(f"{src_info['path']}/**/images", recursive=True)
        lbl_dirs = glob.glob(f"{src_info['path']}/**/labels", recursive=True)

        if img_dirs and lbl_dirs:
            n = generate_synthetic_from_source(
                img_dirs[0], lbl_dirs[0], SYNTH_OUTPUT,
                max_per_transform=100
            )
            synth_total += n
            print(f"  Generated {n} synthetic images from {src_name}")

log_source('synthetic', synth_total > 0, synth_total, SYNTH_OUTPUT)

# Visualize synthetic examples
synth_imgs = glob.glob(f"{SYNTH_OUTPUT}/images/*.jpg")[:6]
if synth_imgs:
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for ax, ip in zip(axes.flatten(), synth_imgs):
        img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(Path(ip).stem.split('_')[-1], fontsize=9)
        ax.axis('off')
    plt.suptitle('Synthetic Augmented Samples', fontweight='bold')
    plt.tight_layout()
    plt.show()

print(f"\n✅ Total synthetic images generated: {synth_total}")

In [ ]:
# 2E — Dataset Source Summary
print("=" * 55)
print(" DATASET ASSEMBLY SUMMARY")
print("=" * 55)
total_collected = 0
for src, info in SOURCE_STATUS.items():
    status = '✅' if info['success'] else '❌'
    print(f"  {status} {src:<15} {info['images']:>6} images")
    total_collected += info['images']
print("-" * 55)
print(f"  {'TOTAL':<15} {total_collected:>6} images")
print("=" * 55)

if total_collected == 0:
    print("\n⚠️  WARNING: No images downloaded from any source!")
    print("   Please configure at least one API key and re-run.")
else:
    print(f"\n✅ Multi-source dataset assembled: {total_collected} images total.")

---
# Section 3: Dataset Consolidation & Preprocessing
**Per the paper:** Merge all sources into a unified YOLO format, remap class labels, create stratified splits, and generate 5-fold cross-validation YAML files.

In [ ]:
# 3.1 — Class Label Remapping
# Paper classes: 0=Handgun, 1=Knife, 2=Rifle, 3=Shotgun
# Different sources use different class IDs → remap to unified scheme

# Unified class map
UNIFIED_CLASS_MAP = {
    'handgun':  0, 'gun':       0, 'pistol':    0,
    'knife':    1, 'blade':     1, 'dagger':    1,
    'rifle':    2, 'long gun':  2, 'ar':        2,
    'shotgun':  3, 'firearm':   0,  # map generic firearm to handgun
}

# Open Images class mapping
OPEN_IMAGES_CLASS_MAP = {
    'Handgun': 0,
    'Knife':   1,
    'Rifle':   2,
    # Shotgun not in Open Images — will be noted
}

print("Unified class mapping:")
for name, idx in sorted(set(UNIFIED_CLASS_MAP.items()), key=lambda x: x[1]):
    print(f"  {idx}: {name}")

print(f"\nNote: 'Shotgun' (class 3) sourced from Roboflow/Kaggle.")
print(f"       Open Images only provides classes 0, 1, 2.")

In [ ]:
# 3.2 — Merge All Sources into Unified YOLO Directory

UNIFIED_DIR  = f"{LOCAL_DATA}/unified"
UNI_IMG_DIR  = f"{UNIFIED_DIR}/images"
UNI_LBL_DIR  = f"{UNIFIED_DIR}/labels"
os.makedirs(UNI_IMG_DIR, exist_ok=True)
os.makedirs(UNI_LBL_DIR, exist_ok=True)

def find_image_label_pairs(base_dir):
    """Find all (image_path, label_path) pairs in a directory tree."""
    pairs = []
    img_files = glob.glob(f"{base_dir}/**/*.jpg", recursive=True)
    img_files += glob.glob(f"{base_dir}/**/*.png", recursive=True)
    img_files += glob.glob(f"{base_dir}/**/*.jpeg", recursive=True)

    for img_path in img_files:
        # Try to find corresponding label
        stem  = Path(img_path).stem
        # Check adjacent labels folder
        lbl1  = Path(img_path).parent.parent / 'labels' / (stem + '.txt')
        lbl2  = Path(img_path).parent / (stem + '.txt')
        lbl3  = Path(str(img_path).replace('/images/', '/labels/').replace('.jpg', '.txt').replace('.png', '.txt'))

        lbl_path = None
        for candidate in [lbl1, lbl2, lbl3]:
            if candidate.exists():
                lbl_path = str(candidate)
                break

        pairs.append((img_path, lbl_path))
    return pairs


def remap_label_file(lbl_path, src_class_names=None, new_lbl_path=None):
    """
    Read a YOLO label file and remap class IDs to the unified scheme.
    src_class_names: list of class names for the source dataset (index = class ID)
    """
    if lbl_path is None or not os.path.exists(lbl_path):
        return False

    try:
        with open(lbl_path, 'r') as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            src_id = int(parts[0])

            # Try to get unified class ID
            new_id = None
            if src_class_names and src_id < len(src_class_names):
                cls_name = src_class_names[src_id].lower()
                new_id   = UNIFIED_CLASS_MAP.get(cls_name)
            else:
                # If source class IDs already map 0-3, keep them
                if src_id <= 3:
                    new_id = src_id

            if new_id is not None:
                parts[0] = str(new_id)
                # Validate bbox values
                coords = list(map(float, parts[1:5]))
                if all(0.0 <= c <= 1.0 for c in coords):
                    new_lines.append(' '.join(parts) + '\n')

        if new_lbl_path:
            with open(new_lbl_path, 'w') as f:
                f.writelines(new_lines)
        return len(new_lines) > 0

    except Exception:
        return False


# Merge all raw sources
all_pairs   = []
raw_sources = [
    (f"{LOCAL_DATA}/raw/roboflow",   None),
    (f"{LOCAL_DATA}/raw/openimages", ['Handgun', 'Knife', 'Rifle']),
    (f"{LOCAL_DATA}/raw/kaggle",     None),
    (f"{LOCAL_DATA}/raw/synthetic",  None),
]

total_merged = 0
for src_dir, src_classes in raw_sources:
    if not os.path.exists(src_dir):
        continue

    pairs = find_image_label_pairs(src_dir)
    print(f"  Found {len(pairs)} images in {Path(src_dir).name}")

    for img_path, lbl_path in tqdm(pairs, desc=f"  Merging {Path(src_dir).name}"):
        try:
            stem    = Path(img_path).stem
            src_tag = Path(src_dir).name[:3].upper()
            new_stem = f"{src_tag}_{stem}"

            # Copy image
            ext      = Path(img_path).suffix
            out_img  = f"{UNI_IMG_DIR}/{new_stem}{ext}"
            shutil.copy2(img_path, out_img)

            # Remap and copy label
            out_lbl = f"{UNI_LBL_DIR}/{new_stem}.txt"
            ok = remap_label_file(lbl_path, src_classes, out_lbl)

            if ok:
                all_pairs.append((out_img, out_lbl))
                total_merged += 1
        except Exception:
            continue

print(f"\n✅ Unified dataset: {total_merged} valid image-label pairs")

In [ ]:
# 3.3 — Dataset Quality Check
# Verify label files, validate bounding box coordinates

print("Running dataset quality checks...")

class_counts    = Counter()
invalid_images  = []
invalid_labels  = []
valid_pairs     = []
img_sizes       = []

for img_path, lbl_path in tqdm(all_pairs, desc="Quality checking"):
    img_ok = True
    lbl_ok = True

    # Check image is readable
    try:
        img  = cv2.imread(img_path)
        if img is None:
            img_ok = False
        else:
            h, w = img.shape[:2]
            img_sizes.append((w, h))
    except Exception:
        img_ok = False

    # Check label exists and has valid boxes
    if not lbl_path or not os.path.exists(lbl_path):
        lbl_ok = False
    else:
        try:
            with open(lbl_path, 'r') as f:
                lines = f.readlines()

            if not lines:
                lbl_ok = False
            else:
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(parts[0])
                        coords = list(map(float, parts[1:5]))
                        if 0 <= cls_id <= 3 and all(0.0 <= c <= 1.0 for c in coords):
                            class_counts[cls_id] += 1
                        else:
                            lbl_ok = False
        except Exception:
            lbl_ok = False

    if img_ok and lbl_ok:
        valid_pairs.append((img_path, lbl_path))
    else:
        if not img_ok:  invalid_images.append(img_path)
        if not lbl_ok:  invalid_labels.append(lbl_path)

print(f"\n{'='*50}")
print(f" QUALITY CHECK RESULTS")
print(f"{'='*50}")
print(f"  Valid pairs:   {len(valid_pairs):>6}")
print(f"  Invalid imgs:  {len(invalid_images):>6}")
print(f"  Invalid lbls:  {len(invalid_labels):>6}")
print(f"\n  Per-class annotations:")
for cls_id, cls_name in enumerate(CLASS_NAMES):
    print(f"    {cls_id} - {cls_name:<10}: {class_counts.get(cls_id, 0):>6} boxes")

all_pairs = valid_pairs
print(f"\n✅ Using {len(all_pairs)} valid samples for training.")

In [ ]:
# 3.4 — Stratified 70/15/15 Train/Val/Test Split

from sklearn.model_selection import train_test_split

# Build per-image dominant class label (for stratification)
def get_dominant_class(lbl_path):
    """Return the most frequent class in a label file."""
    try:
        with open(lbl_path, 'r') as f:
            ids = [int(l.split()[0]) for l in f if l.strip()]
        return Counter(ids).most_common(1)[0][0] if ids else -1
    except Exception:
        return -1

print("Computing per-image dominant class for stratification...")
dominant_classes = [
    get_dominant_class(lbl) for _, lbl in tqdm(all_pairs, desc="Stratifying")
]

# Filter out -1 (unreadable)
valid_idx        = [i for i, c in enumerate(dominant_classes) if c >= 0]
filtered_pairs   = [all_pairs[i] for i in valid_idx]
filtered_classes = [dominant_classes[i] for i in valid_idx]

print(f"  Stratifying {len(filtered_pairs)} samples...")

# Split: 70% train+val, 15% test
trainval_pairs, test_pairs, trainval_cls, test_cls = train_test_split(
    filtered_pairs, filtered_classes,
    test_size=TEST_RATIO, random_state=42, stratify=filtered_classes
)

# Split trainval: 70% train, 15% val (from 85% → ~82%/18% relative)
val_relative = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
train_pairs, val_pairs, train_cls, val_cls = train_test_split(
    trainval_pairs, trainval_cls,
    test_size=val_relative, random_state=42, stratify=trainval_cls
)

print(f"\n{'='*45}")
print(f" DATASET SPLIT (Paper: 70/15/15)")
print(f"{'='*45}")
print(f"  Train:  {len(train_pairs):>6} images ({len(train_pairs)/len(filtered_pairs)*100:.1f}%)")
print(f"  Val:    {len(val_pairs):>6} images ({len(val_pairs)/len(filtered_pairs)*100:.1f}%)")
print(f"  Test:   {len(test_pairs):>6} images ({len(test_pairs)/len(filtered_pairs)*100:.1f}%)")
print(f"  Total:  {len(filtered_pairs):>6} images")

In [ ]:
# 3.5 — Copy Split Files to Structured Directories

SPLITS = {
    'train': train_pairs,
    'val':   val_pairs,
    'test':  test_pairs,
}

SPLIT_BASE = f"{LOCAL_DATA}/splits"

for split_name, pairs in SPLITS.items():
    img_dir = f"{SPLIT_BASE}/{split_name}/images"
    lbl_dir = f"{SPLIT_BASE}/{split_name}/labels"
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    for img_path, lbl_path in tqdm(pairs, desc=f"Copying {split_name}"):
        try:
            shutil.copy2(img_path, img_dir)
            shutil.copy2(lbl_path, lbl_dir)
        except Exception:
            continue

    print(f"  ✅ {split_name}: {len(pairs)} images copied")

# Write main dataset YAML
main_yaml = {
    'path':  SPLIT_BASE,
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc':    4,
    'names': CLASS_NAMES,
}
with open(f"{LOCAL_DATA}/dataset.yaml", 'w') as f:
    yaml.dump(main_yaml, f, default_flow_style=False)

print(f"\n✅ Main dataset.yaml written: {LOCAL_DATA}/dataset.yaml")

In [ ]:
# 3.6 — Create 5-Fold Cross-Validation Splits
# Paper: 5-fold CV, 80/20 train/val per fold, on training set only

os.makedirs(LOCAL_FOLDS, exist_ok=True)

# Use train+val pairs for cross-validation
cv_pairs   = train_pairs + val_pairs
cv_classes = train_cls   + val_cls

skf  = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)
fold_yamls = []

print(f"Creating {NUM_FOLDS}-fold CV splits (80/20 per fold)...")

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(cv_pairs, cv_classes), 1):
    fold_dir    = f"{LOCAL_FOLDS}/fold{fold_idx}"
    f_train_img = f"{fold_dir}/train/images"
    f_train_lbl = f"{fold_dir}/train/labels"
    f_val_img   = f"{fold_dir}/val/images"
    f_val_lbl   = f"{fold_dir}/val/labels"

    for d in [f_train_img, f_train_lbl, f_val_img, f_val_lbl]:
        os.makedirs(d, exist_ok=True)

    # Copy fold images/labels
    for i in tqdm(train_idx, desc=f"Fold {fold_idx} train", leave=False):
        img_p, lbl_p = cv_pairs[i]
        try:
            shutil.copy2(img_p, f_train_img)
            shutil.copy2(lbl_p, f_train_lbl)
        except Exception: pass

    for i in tqdm(val_idx, desc=f"Fold {fold_idx} val", leave=False):
        img_p, lbl_p = cv_pairs[i]
        try:
            shutil.copy2(img_p, f_val_img)
            shutil.copy2(lbl_p, f_val_lbl)
        except Exception: pass

    # Write fold YAML
    fold_yaml_path = f"{fold_dir}/data_fold{fold_idx}.yaml"
    fold_yaml = {
        'path':  fold_dir,
        'train': 'train/images',
        'val':   'val/images',
        'nc':    4,
        'names': CLASS_NAMES,
    }
    with open(fold_yaml_path, 'w') as f:
        yaml.dump(fold_yaml, f, default_flow_style=False)

    fold_yamls.append(fold_yaml_path)

    n_train = len(train_idx)
    n_val   = len(val_idx)
    print(f"  Fold {fold_idx}: {n_train} train / {n_val} val → {fold_yaml_path}")

print(f"\n✅ {NUM_FOLDS} fold YAML files created.")

In [ ]:
# 3.7 — Dataset Statistics & Visualization

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Dataset Statistics', fontsize=14, fontweight='bold')

# 1. Per-class distribution
ax1 = axes[0]
class_totals = [class_counts.get(i, 0) for i in range(4)]
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']
bars = ax1.bar(CLASS_NAMES, class_totals, color=colors, edgecolor='black', linewidth=0.8)
ax1.set_title('Per-Class Annotation Count')
ax1.set_ylabel('# Bounding Boxes')
ax1.set_xlabel('Weapon Class')
for bar, val in zip(bars, class_totals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(val), ha='center', va='bottom', fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# 2. Split distribution
ax2 = axes[1]
split_sizes = [len(train_pairs), len(val_pairs), len(test_pairs)]
split_labels = ['Train', 'Val', 'Test']
split_colors = ['#27AE60', '#2980B9', '#E74C3C']
wedges, texts, autotexts = ax2.pie(
    split_sizes, labels=split_labels, colors=split_colors,
    autopct='%1.1f%%', startangle=90, pctdistance=0.85
)
for t in autotexts: t.set_fontsize(10)
ax2.set_title('Dataset Split Distribution')

# 3. Source distribution
ax3 = axes[2]
src_names  = [k for k, v in SOURCE_STATUS.items() if v['success']]
src_counts = [SOURCE_STATUS[k]['images'] for k in src_names]
if src_names:
    src_colors = ['#8E44AD', '#16A085', '#D35400', '#C0392B']
    ax3.bar(src_names, src_counts, color=src_colors[:len(src_names)],
            edgecolor='black', linewidth=0.8)
    ax3.set_title('Images per Source')
    ax3.set_ylabel('# Images')
    ax3.set_xlabel('Data Source')
    ax3.grid(axis='y', alpha=0.3)
else:
    ax3.text(0.5, 0.5, 'No sources\nsuccessful', ha='center', va='center',
             transform=ax3.transAxes, fontsize=12)

plt.tight_layout()
stats_fig_path = f"{DRIVE_BASE}/results/dataset_statistics.png"
plt.savefig(stats_fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Statistics saved to Drive: {stats_fig_path}")

---
# Section 4: Ablation Study — Train 3 YOLOv8 Variants
**Per the paper:** Train YOLOv8n, YOLOv8s, and YOLOv8m on Fold 1 data using exact paper hyperparameters. Compare mAP@50, mAP@50:95, FPS, and parameter count.

In [ ]:
# 4.1 — Define Paper-Exact Training Hyperparameters

TRAIN_KWARGS = dict(
    # Optimization (Paper-exact)
    optimizer     = OPTIMIZER,
    momentum      = MOMENTUM,
    weight_decay  = WEIGHT_DECAY,
    lr0           = LR0,
    lrf           = LRF / LR0,   # lrf is ratio, not absolute
    cos_lr        = True,         # Cosine annealing LR
    warmup_epochs = WARMUP_EPOCHS,

    # Training schedule
    epochs        = EPOCHS,
    patience      = PATIENCE,
    batch         = BATCH,
    imgsz         = IMGSZ,
    device        = DEVICE,
    workers       = 2,
    seed          = 42,

    # Augmentations (Paper-exact)
    mosaic        = AUG_MOSAIC,
    fliplr        = AUG_FLIPLR,
    scale         = AUG_SCALE,
    translate     = AUG_TRANSLATE,
    hsv_h         = AUG_HSV_H,
    hsv_s         = AUG_HSV_S,
    hsv_v         = AUG_HSV_V,
    erasing       = AUG_ERASING,
    mixup         = AUG_MIXUP,

    # NMS
    iou           = NMS_IOU_THRESH,
    conf          = CONF_THRESH,

    # Misc
    save          = True,
    save_period   = 10,
    verbose       = True,
    plots         = True,
)

print("Training hyperparameters (Paper-Exact):")
for k, v in TRAIN_KWARGS.items():
    print(f"  {k:<18}: {v}")

In [ ]:
# 4.2 — Ablation Study: Train YOLOv8n, YOLOv8s, YOLOv8m on Fold 1

FOLD1_YAML      = fold_yamls[0]  # Fold 1 YAML
ablation_results = []

print("=" * 60)
print(" ABLATION STUDY — 3 Model Variants")
print(" Dataset: Fold 1")
print("=" * 60)

for model_name, model_weights in MODELS_TO_TRAIN:
    print(f"\n{'─'*60}")
    print(f" Training: {model_name.upper()} ({model_weights})")
    print(f"{'─'*60}")

    run_dir = f"{LOCAL_RUNS}/ablation/{model_name}"
    os.makedirs(run_dir, exist_ok=True)

    t_start = time.time()

    try:
        model = YOLO(model_weights)
        results = model.train(
            data    = FOLD1_YAML,
            name    = f"ablation_{model_name}",
            project = run_dir,
            **TRAIN_KWARGS
        )

        t_elapsed = time.time() - t_start

        # Extract metrics
        metrics = model.val(data=FOLD1_YAML, conf=CONF_THRESH, iou=NMS_IOU_THRESH)

        map50    = float(metrics.box.map50)
        map5095  = float(metrics.box.map)
        prec     = float(metrics.box.mp)
        rec      = float(metrics.box.mr)

        # FPS estimation (inference on val set)
        val_imgs = glob.glob(f"{LOCAL_FOLDS}/fold1/val/images/*.jpg")[:50]
        fps_start = time.time()
        if val_imgs:
            for img in val_imgs:
                model.predict(img, conf=CONF_THRESH, iou=NMS_IOU_THRESH, verbose=False)
            fps = len(val_imgs) / (time.time() - fps_start)
        else:
            fps = 0.0

        # Model params
        n_params = sum(p.numel() for p in model.model.parameters()) / 1e6

        ablation_results.append({
            'Model':      model_name,
            'Weights':    model_weights,
            'Params(M)':  round(n_params, 2),
            'mAP@50':     round(map50, 4),
            'mAP@50:95':  round(map5095, 4),
            'Precision':  round(prec, 4),
            'Recall':     round(rec, 4),
            'FPS':        round(fps, 1),
            'Time(h)':    round(t_elapsed / 3600, 2),
        })

        # Save model to Drive
        best_pt  = glob.glob(f"{run_dir}/**/best.pt", recursive=True)
        if best_pt:
            drive_wt = f"{DRIVE_BASE}/weights/{model_name}_best.pt"
            shutil.copy2(best_pt[0], drive_wt)
            print(f"  ✅ Saved to Drive: {drive_wt}")

        print(f"  mAP@50: {map50:.4f}  |  mAP@50:95: {map5095:.4f}  |  FPS: {fps:.1f}")

    except Exception as e:
        print(f"  ⚠️  Training failed for {model_name}: {e}")
        ablation_results.append({
            'Model': model_name, 'Weights': model_weights,
            'Params(M)': 0, 'mAP@50': 0, 'mAP@50:95': 0,
            'Precision': 0, 'Recall': 0, 'FPS': 0, 'Time(h)': 0,
            'Error': str(e)
        })

print("\n✅ Ablation training complete.")

In [ ]:
# 4.3 — Ablation Results Table & Model Selection

df_ablation = pd.DataFrame(ablation_results)

print("\n" + "=" * 75)
print(" ABLATION STUDY RESULTS")
print("=" * 75)
print(df_ablation.to_string(index=False))
print("=" * 75)

# Save results
ablation_csv = f"{DRIVE_BASE}/results/ablation_results.csv"
df_ablation.to_csv(ablation_csv, index=False)
print(f"\n✅ Saved ablation results: {ablation_csv}")

# Select best model by mAP@50
if not df_ablation.empty and 'mAP@50' in df_ablation.columns:
    best_row = df_ablation.loc[df_ablation['mAP@50'].idxmax()]
    BEST_MODEL_NAME  = best_row['Model']
    BEST_MODEL_WT    = best_row['Weights']
    print(f"\n🏆 Best model selected: {BEST_MODEL_NAME} (mAP@50 = {best_row['mAP@50']:.4f})")
    print(f"   (Paper predicts YOLOv8s as best balance of accuracy and speed)")
else:
    BEST_MODEL_NAME = 'yolov8s'
    BEST_MODEL_WT   = 'yolov8s.pt'
    print(f"\n  Using paper-specified best model: {BEST_MODEL_NAME}")

# Ablation bar chart
if len(df_ablation) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    metrics_to_plot = ['mAP@50', 'mAP@50:95']
    bar_colors = ['#E74C3C', '#3498DB', '#2ECC71']

    for ax, metric in zip(axes, metrics_to_plot):
        if metric in df_ablation.columns:
            bars = ax.bar(df_ablation['Model'], df_ablation[metric],
                          color=bar_colors[:len(df_ablation)], edgecolor='black')
            ax.set_title(f'Ablation: {metric}', fontweight='bold')
            ax.set_ylabel(metric)
            ax.set_ylim(0, 1)
            ax.grid(axis='y', alpha=0.3)
            for bar, val in zip(bars, df_ablation[metric]):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                        f"{val:.3f}", ha='center', va='bottom', fontsize=9)

    plt.suptitle('Ablation Study — YOLOv8 Variant Comparison', fontweight='bold')
    plt.tight_layout()
    abl_fig = f"{DRIVE_BASE}/results/ablation_comparison.png"
    plt.savefig(abl_fig, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Chart saved: {abl_fig}")

---
# Section 5: 5-Fold Cross-Validation (YOLOv8s)
**Per the paper:** Full 5-fold cross-validation is run with YOLOv8s only (selected from ablation study). Each fold uses fresh pretrained weights. Metrics are averaged across all 5 folds.

In [ ]:
# 5.1 — 5-Fold Cross-Validation Training (YOLOv8s)

CV_MODEL_WT = f"{BEST_MODEL_NAME}.pt"  # Fresh pretrained weights each fold
cv_results  = []

print("=" * 60)
print(f" 5-FOLD CROSS-VALIDATION — {BEST_MODEL_NAME.upper()}")
print("=" * 60)

for fold_idx, fold_yaml in enumerate(fold_yamls, 1):
    print(f"\n{'─'*60}")
    print(f" FOLD {fold_idx}/{NUM_FOLDS}: {fold_yaml}")
    print(f"{'─'*60}")

    fold_run_dir = f"{LOCAL_RUNS}/cv/{BEST_MODEL_NAME}_fold{fold_idx}"
    os.makedirs(fold_run_dir, exist_ok=True)

    try:
        # Load FRESH pretrained weights for each fold
        model = YOLO(CV_MODEL_WT)

        results = model.train(
            data    = fold_yaml,
            name    = f"{BEST_MODEL_NAME}_fold{fold_idx}",
            project = fold_run_dir,
            **TRAIN_KWARGS
        )

        # Validate on fold's val set
        val_metrics = model.val(
            data = fold_yaml,
            conf = CONF_THRESH,
            iou  = NMS_IOU_THRESH,
        )

        # Extract overall metrics
        overall_map50   = float(val_metrics.box.map50)
        overall_map5095 = float(val_metrics.box.map)
        overall_prec    = float(val_metrics.box.mp)
        overall_recall  = float(val_metrics.box.mr)
        f1_score        = 2 * overall_prec * overall_recall / (overall_prec + overall_recall + 1e-8)

        # Extract per-class metrics
        per_class = {}
        if hasattr(val_metrics.box, 'ap_class_index'):
            for ci, cls_idx in enumerate(val_metrics.box.ap_class_index):
                cls_name = CLASS_NAMES[cls_idx] if cls_idx < len(CLASS_NAMES) else f'cls{cls_idx}'
                per_class[cls_name] = {
                    'AP50': float(val_metrics.box.ap50[ci]) if ci < len(val_metrics.box.ap50) else 0,
                    'P':    float(val_metrics.box.p[ci])    if ci < len(val_metrics.box.p)    else 0,
                    'R':    float(val_metrics.box.r[ci])    if ci < len(val_metrics.box.r)    else 0,
                }

        fold_result = {
            'Fold':       fold_idx,
            'mAP@50':     round(overall_map50, 4),
            'mAP@50:95':  round(overall_map5095, 4),
            'Precision':  round(overall_prec, 4),
            'Recall':     round(overall_recall, 4),
            'F1':         round(f1_score, 4),
        }

        # Add per-class metrics
        for cls_name in CLASS_NAMES:
            pc = per_class.get(cls_name, {'AP50': 0, 'P': 0, 'R': 0})
            fold_result[f'{cls_name}_AP50'] = round(pc['AP50'], 4)
            fold_result[f'{cls_name}_P']    = round(pc['P'],    4)
            fold_result[f'{cls_name}_R']    = round(pc['R'],    4)

        cv_results.append(fold_result)

        # Save fold best weights to Drive
        best_pt = glob.glob(f"{fold_run_dir}/**/best.pt", recursive=True)
        if best_pt:
            drive_wt = f"{DRIVE_BASE}/weights/{BEST_MODEL_NAME}_fold{fold_idx}_best.pt"
            shutil.copy2(best_pt[0], drive_wt)

        print(f"  ✅ Fold {fold_idx}: mAP@50={overall_map50:.4f} | mAP@50:95={overall_map5095:.4f} | F1={f1_score:.4f}")

    except Exception as e:
        print(f"  ⚠️  Fold {fold_idx} failed: {e}")
        cv_results.append({'Fold': fold_idx, 'mAP@50': 0, 'mAP@50:95': 0,
                            'Precision': 0, 'Recall': 0, 'F1': 0, 'Error': str(e)})
        continue

print("\n✅ 5-Fold Cross-Validation complete.")

In [ ]:
# 5.2 — Cross-Validation Results Summary

df_cv = pd.DataFrame(cv_results)

# Compute mean ± std across folds
numeric_cols = df_cv.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'Fold']

cv_mean = df_cv[numeric_cols].mean()
cv_std  = df_cv[numeric_cols].std()

print("\n" + "=" * 70)
print(f" 5-FOLD CROSS-VALIDATION RESULTS — {BEST_MODEL_NAME.upper()}")
print("=" * 70)
print(df_cv[['Fold', 'mAP@50', 'mAP@50:95', 'Precision', 'Recall', 'F1']].to_string(index=False))
print("-" * 70)
print(f"  {'MEAN':<10}: mAP@50={cv_mean['mAP@50']:.4f} | "
      f"mAP@50:95={cv_mean['mAP@50:95']:.4f} | "
      f"F1={cv_mean['F1']:.4f}")
print(f"  {'STD':<10}: mAP@50={cv_std['mAP@50']:.4f}  | "
      f"mAP@50:95={cv_std['mAP@50:95']:.4f}  | "
      f"F1={cv_std['F1']:.4f}")
print("=" * 70)

# Per-class averages
print("\n Per-Class Average AP@50 across 5 Folds:")
for cls_name in CLASS_NAMES:
    col = f'{cls_name}_AP50'
    if col in df_cv.columns:
        print(f"   {cls_name:<10}: {df_cv[col].mean():.4f} ± {df_cv[col].std():.4f}")

# Save CV results
cv_csv = f"{DRIVE_BASE}/results/cross_validation_results.csv"
df_cv.to_csv(cv_csv, index=False)
print(f"\n✅ CV results saved: {cv_csv}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# mAP per fold
ax1 = axes[0]
folds = df_cv['Fold'].tolist()
ax1.plot(folds, df_cv['mAP@50'],    marker='o', label='mAP@50',    color='#E74C3C', linewidth=2)
ax1.plot(folds, df_cv['mAP@50:95'], marker='s', label='mAP@50:95', color='#3498DB', linewidth=2)
ax1.axhline(cv_mean['mAP@50'],    color='#E74C3C', linestyle='--', alpha=0.5, label=f"Mean mAP@50={cv_mean['mAP@50']:.3f}")
ax1.axhline(cv_mean['mAP@50:95'], color='#3498DB', linestyle='--', alpha=0.5, label=f"Mean mAP@50:95={cv_mean['mAP@50:95']:.3f}")
ax1.set_xlabel('Fold')
ax1.set_ylabel('mAP')
ax1.set_title('mAP per Fold')
ax1.legend(fontsize=8)
ax1.set_xticks(folds)
ax1.grid(alpha=0.3)

# Per-class AP across folds
ax2 = axes[1]
cls_means = []
cls_stds  = []
cls_valid  = []
for cls_name in CLASS_NAMES:
    col = f'{cls_name}_AP50'
    if col in df_cv.columns:
        cls_means.append(df_cv[col].mean())
        cls_stds.append(df_cv[col].std())
        cls_valid.append(cls_name)

if cls_valid:
    cls_colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']
    bars = ax2.bar(cls_valid, cls_means, yerr=cls_stds, capsize=5,
                   color=cls_colors[:len(cls_valid)], edgecolor='black')
    ax2.set_title('Per-Class AP@50 (Mean ± Std, 5 Folds)')
    ax2.set_ylabel('AP@50')
    ax2.set_ylim(0, 1)
    ax2.grid(axis='y', alpha=0.3)

plt.suptitle(f'5-Fold Cross-Validation — {BEST_MODEL_NAME.upper()}', fontweight='bold')
plt.tight_layout()
cv_fig = f"{DRIVE_BASE}/results/cross_validation_plot.png"
plt.savefig(cv_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ CV plot saved: {cv_fig}")

---
# Section 6: Per-Class Evaluation & Adverse Condition Testing
**Per the paper:** Evaluate best model on held-out test set under normal and 4 adverse conditions:
1. Normal (baseline)
2. Low illumination (gamma darkening)
3. Motion blur
4. Partial occlusion
5. All combined

In [ ]:
# 6.1 — Load Best Model (YOLOv8s, best fold)
# Try to load from Drive weights, fallback to local

best_weight_path = None

# Try Drive weights first
drive_weights = glob.glob(f"{DRIVE_BASE}/weights/{BEST_MODEL_NAME}*.pt")
if drive_weights:
    best_weight_path = drive_weights[0]
    print(f"✅ Loaded best model from Drive: {best_weight_path}")
else:
    # Try local runs
    local_weights = glob.glob(f"{LOCAL_RUNS}/**/{BEST_MODEL_NAME}*/best.pt", recursive=True)
    if local_weights:
        best_weight_path = local_weights[0]
        print(f"✅ Loaded best model from local: {best_weight_path}")
    else:
        # Fallback to pretrained
        best_weight_path = f"{BEST_MODEL_NAME}.pt"
        print(f"⚠️  No trained weights found. Using pretrained: {best_weight_path}")
        print(f"   (Evaluation metrics will reflect pretrained model, not custom-trained)")

best_model = YOLO(best_weight_path)
print(f"   Model loaded: {type(best_model).__name__}")

In [ ]:
# 6.2 — Baseline Evaluation on Test Set

TEST_IMG_DIR = f"{SPLIT_BASE}/test/images"
TEST_LBL_DIR = f"{SPLIT_BASE}/test/labels"

# Write test YAML
test_yaml_path = f"{LOCAL_DATA}/test.yaml"
test_yaml = {
    'path':  SPLIT_BASE,
    'train': 'train/images',
    'val':   'test/images',   # Evaluate on test set
    'nc':    4,
    'names': CLASS_NAMES,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml, f, default_flow_style=False)

print("Evaluating on held-out test set (baseline — normal conditions)...")

try:
    baseline_metrics = best_model.val(
        data = test_yaml_path,
        conf = CONF_THRESH,
        iou  = NMS_IOU_THRESH,
        split = 'val',
    )

    class_metrics = []
    if hasattr(baseline_metrics.box, 'ap_class_index'):
        for ci, cls_idx in enumerate(baseline_metrics.box.ap_class_index):
            cls_name = CLASS_NAMES[cls_idx] if cls_idx < len(CLASS_NAMES) else f'class_{cls_idx}'
            ap50  = float(baseline_metrics.box.ap50[ci]) if ci < len(baseline_metrics.box.ap50) else 0
            p_val = float(baseline_metrics.box.p[ci])    if ci < len(baseline_metrics.box.p)    else 0
            r_val = float(baseline_metrics.box.r[ci])    if ci < len(baseline_metrics.box.r)    else 0
            f1    = 2 * p_val * r_val / (p_val + r_val + 1e-8)
            class_metrics.append({
                'Class':     cls_name,
                'AP@50':     round(ap50, 4),
                'Precision': round(p_val, 4),
                'Recall':    round(r_val, 4),
                'F1':        round(f1, 4),
            })

    df_class = pd.DataFrame(class_metrics)

    print(f"\n{'='*55}")
    print(f" PER-CLASS METRICS — Test Set (Normal Conditions)")
    print(f"{'='*55}")
    print(df_class.to_string(index=False))
    print(f"{'─'*55}")
    print(f"  Overall mAP@50:    {float(baseline_metrics.box.map50):.4f}")
    print(f"  Overall mAP@50:95: {float(baseline_metrics.box.map):.4f}")

    class_csv = f"{DRIVE_BASE}/results/class_metrics.csv"
    df_class.to_csv(class_csv, index=False)
    print(f"\n✅ Per-class metrics saved: {class_csv}")

except Exception as e:
    print(f"⚠️  Test evaluation failed: {e}")
    df_class = pd.DataFrame()
    baseline_metrics = None

In [ ]:
# 6.3 — Adverse Condition Testing
# Apply augmentations to test images and re-evaluate

def create_augmented_test_split(test_img_dir, test_lbl_dir, aug_name, transform_fn, base_dir):
    """
    Create a temporary test split with augmented images.
    Returns path to YAML file for evaluation.
    """
    aug_img_dir = f"{base_dir}/test_aug_{aug_name}/images"
    aug_lbl_dir = f"{base_dir}/test_aug_{aug_name}/labels"
    os.makedirs(aug_img_dir, exist_ok=True)
    os.makedirs(aug_lbl_dir, exist_ok=True)

    img_files = glob.glob(f"{test_img_dir}/*.jpg")
    img_files += glob.glob(f"{test_img_dir}/*.png")

    for img_path in tqdm(img_files, desc=f"  Applying {aug_name}", leave=False):
        try:
            img = cv2.imread(img_path)
            if img is None: continue
            aug_img = transform_fn(img)
            stem    = Path(img_path).stem
            cv2.imwrite(f"{aug_img_dir}/{stem}.jpg", aug_img)

            lbl_path = f"{test_lbl_dir}/{stem}.txt"
            if os.path.exists(lbl_path):
                shutil.copy2(lbl_path, f"{aug_lbl_dir}/{stem}.txt")
        except Exception:
            continue

    # Write YAML
    aug_yaml_path = f"{base_dir}/test_aug_{aug_name}.yaml"
    aug_yaml = {
        'path':  f"{base_dir}/test_aug_{aug_name}",
        'val':   'images',
        'nc':    4,
        'names': CLASS_NAMES,
    }
    with open(aug_yaml_path, 'w') as f:
        yaml.dump(aug_yaml, f, default_flow_style=False)

    return aug_yaml_path


def combined_augmentation(image_bgr):
    img = apply_gamma_darkening(image_bgr)
    img = apply_motion_blur(img)
    img = apply_occlusion(img)
    return img


CONDITIONS = [
    ('Normal',          None),
    ('LowIllumination', apply_gamma_darkening),
    ('MotionBlur',      apply_motion_blur),
    ('Occlusion',       apply_occlusion),
    ('Combined',        combined_augmentation),
]

benchmark_results = []
AUG_TMP = f"{LOCAL_BASE}/aug_test_tmp"
os.makedirs(AUG_TMP, exist_ok=True)

print("Adverse Condition Benchmarking...")

for cond_name, transform_fn in CONDITIONS:
    print(f"\n  Testing condition: {cond_name}")

    try:
        if transform_fn is None:
            # Normal: use existing test YAML
            eval_yaml = test_yaml_path
        else:
            eval_yaml = create_augmented_test_split(
                TEST_IMG_DIR, TEST_LBL_DIR, cond_name, transform_fn, AUG_TMP
            )

        metrics = best_model.val(
            data  = eval_yaml,
            conf  = CONF_THRESH,
            iou   = NMS_IOU_THRESH,
            split = 'val',
        )

        map50   = float(metrics.box.map50)
        map5095 = float(metrics.box.map)
        prec    = float(metrics.box.mp)
        rec     = float(metrics.box.mr)
        f1      = 2 * prec * rec / (prec + rec + 1e-8)

        benchmark_results.append({
            'Condition':  cond_name,
            'mAP@50':     round(map50, 4),
            'mAP@50:95':  round(map5095, 4),
            'Precision':  round(prec, 4),
            'Recall':     round(rec, 4),
            'F1':         round(f1, 4),
        })

        print(f"    mAP@50={map50:.4f}  |  F1={f1:.4f}")

    except Exception as e:
        print(f"    ⚠️  Condition {cond_name} failed: {e}")
        benchmark_results.append({'Condition': cond_name, 'Error': str(e)})

print("\n✅ Adverse condition testing complete.")

In [ ]:
# 6.4 — Benchmark Results Table & Visualization

df_bench = pd.DataFrame(benchmark_results)

print("\n" + "=" * 65)
print(" ADVERSE CONDITION BENCHMARK RESULTS")
print("=" * 65)
if 'mAP@50' in df_bench.columns:
    print(df_bench[['Condition', 'mAP@50', 'mAP@50:95', 'Precision', 'Recall', 'F1']].to_string(index=False))
print("=" * 65)

# Save benchmark CSV
bench_csv = f"{DRIVE_BASE}/results/benchmark_results.csv"
df_bench.to_csv(bench_csv, index=False)
print(f"\n✅ Benchmark results saved: {bench_csv}")

# Visualization
if 'mAP@50' in df_bench.columns and len(df_bench) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # mAP@50 by condition
    ax1 = axes[0]
    palette = ['#27AE60', '#E74C3C', '#F39C12', '#8E44AD', '#E67E22']
    bars = ax1.bar(
        df_bench['Condition'], df_bench['mAP@50'],
        color=palette[:len(df_bench)], edgecolor='black'
    )
    ax1.set_title('mAP@50 by Condition', fontweight='bold')
    ax1.set_ylabel('mAP@50')
    ax1.set_xticklabels(df_bench['Condition'], rotation=20, ha='right')
    ax1.set_ylim(0, 1)
    ax1.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, df_bench['mAP@50']):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{val:.3f}", ha='center', fontsize=8)

    # Radar chart — P/R/F1 across conditions
    ax2 = axes[1]
    if 'F1' in df_bench.columns:
        ax2.bar(df_bench['Condition'], df_bench['F1'],
                color=palette[:len(df_bench)], edgecolor='black', alpha=0.8)
        ax2.set_title('F1 Score by Condition', fontweight='bold')
        ax2.set_ylabel('F1 Score')
        ax2.set_xticklabels(df_bench['Condition'], rotation=20, ha='right')
        ax2.set_ylim(0, 1)
        ax2.grid(axis='y', alpha=0.3)

    plt.suptitle('Adverse Condition Robustness Evaluation', fontweight='bold')
    plt.tight_layout()
    bench_fig = f"{DRIVE_BASE}/results/benchmark_plot.png"
    plt.savefig(bench_fig, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Benchmark plot saved: {bench_fig}")

# Cleanup temporary augmentation directories
try:
    shutil.rmtree(AUG_TMP)
except Exception:
    pass

---
# Section 7: Export, Download & Project Integration
Export best model weights, package all results, and provide instructions for using the model in the local project.

In [ ]:
# 7.1 — Select Final Best Model Weights
# Choose the fold with highest mAP@50 from cross-validation

final_weights = None

if not df_cv.empty and 'mAP@50' in df_cv.columns:
    best_fold_row = df_cv.loc[df_cv['mAP@50'].idxmax()]
    best_fold_num = int(best_fold_row['Fold'])
    print(f"✅ Best fold: {best_fold_num} (mAP@50 = {best_fold_row['mAP@50']:.4f})")

    # Look for that fold's weights on Drive
    cand = f"{DRIVE_BASE}/weights/{BEST_MODEL_NAME}_fold{best_fold_num}_best.pt"
    if os.path.exists(cand):
        final_weights = cand
        print(f"   Using fold weights: {final_weights}")

if final_weights is None:
    # Fallback: any available weights
    available = glob.glob(f"{DRIVE_BASE}/weights/*.pt")
    if available:
        final_weights = available[0]
        print(f"  Using available weights: {final_weights}")
    else:
        print("⚠️  No trained weights found on Drive.")
        final_weights = f"{BEST_MODEL_NAME}.pt"

print(f"\n📦 Final weights: {final_weights}")

In [ ]:
# 7.2 — Export Model in Multiple Formats

EXPORT_DIR = f"{DRIVE_BASE}/exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

try:
    export_model = YOLO(final_weights)

    # Export to ONNX (for deployment)
    print("Exporting to ONNX...")
    onnx_path = export_model.export(format='onnx', imgsz=IMGSZ, simplify=True)
    if onnx_path:
        shutil.copy2(onnx_path, f"{EXPORT_DIR}/{BEST_MODEL_NAME}_weapon_detector.onnx")
        print(f"  ✅ ONNX saved: {EXPORT_DIR}/{BEST_MODEL_NAME}_weapon_detector.onnx")

    # Copy PyTorch weights
    shutil.copy2(final_weights, f"{EXPORT_DIR}/{BEST_MODEL_NAME}_weapon_detector.pt")
    print(f"  ✅ PyTorch weights: {EXPORT_DIR}/{BEST_MODEL_NAME}_weapon_detector.pt")

    # TorchScript export
    try:
        ts_path = export_model.export(format='torchscript', imgsz=IMGSZ)
        if ts_path:
            shutil.copy2(ts_path, f"{EXPORT_DIR}/{BEST_MODEL_NAME}_weapon_detector.torchscript")
            print(f"  ✅ TorchScript saved")
    except Exception as e:
        print(f"  ⚠️  TorchScript export failed: {e}")

except Exception as e:
    print(f"⚠️  Model export failed: {e}")

In [ ]:
# 7.3 — Download Results CSVs to Local Machine

from google.colab import files
import zipfile

# Package results
results_zip = f"{DRIVE_BASE}/weapon_detection_results.zip"
local_zip   = '/content/weapon_detection_results.zip'

result_files = [
    (f"{DRIVE_BASE}/results/ablation_results.csv",          'ablation_results.csv'),
    (f"{DRIVE_BASE}/results/cross_validation_results.csv",  'cross_validation_results.csv'),
    (f"{DRIVE_BASE}/results/class_metrics.csv",             'class_metrics.csv'),
    (f"{DRIVE_BASE}/results/benchmark_results.csv",         'benchmark_results.csv'),
    (f"{DRIVE_BASE}/results/dataset_statistics.png",        'dataset_statistics.png'),
    (f"{DRIVE_BASE}/results/ablation_comparison.png",       'ablation_comparison.png'),
    (f"{DRIVE_BASE}/results/cross_validation_plot.png",     'cross_validation_plot.png'),
    (f"{DRIVE_BASE}/results/benchmark_plot.png",            'benchmark_plot.png'),
]

with zipfile.ZipFile(local_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for src_path, arc_name in result_files:
        if os.path.exists(src_path):
            zf.write(src_path, arc_name)
            print(f"  Added: {arc_name}")
        else:
            print(f"  ⚠️  Not found: {arc_name}")

# Also add model weights
if final_weights and os.path.exists(final_weights):
    with zipfile.ZipFile(local_zip, 'a', zipfile.ZIP_DEFLATED) as zf:
        zf.write(final_weights, f"{BEST_MODEL_NAME}_best.pt")
    print(f"  Added model weights: {BEST_MODEL_NAME}_best.pt")

print(f"\n✅ Results packaged: {local_zip}")

# Download
print("Downloading results zip to your computer...")
files.download(local_zip)
print("✅ Download started!")

In [ ]:
# 7.4 — Instructions for Local Project Integration

instructions = f"""
╔══════════════════════════════════════════════════════════════════╗
║         LOCAL PROJECT INTEGRATION INSTRUCTIONS                   ║
╚══════════════════════════════════════════════════════════════════╝

1. DOWNLOAD WEIGHTS FROM GOOGLE DRIVE:
   Drive path: {DRIVE_BASE}/exports/{BEST_MODEL_NAME}_weapon_detector.pt
   Or download the zip above and extract the .pt file.

2. COPY TO LOCAL PROJECT:
   Destination: C:\\Users\\Rishabh\\OneDrive\\Desktop\\object detection project\\weights\\
   File: {BEST_MODEL_NAME}_weapon_detector.pt

3. USE IN YOUR DETECTION SCRIPT:

   from ultralytics import YOLO
   import cv2

   model = YOLO(r'weights\\{BEST_MODEL_NAME}_weapon_detector.pt')
   CLASS_NAMES = {CLASS_NAMES}

   # Inference on image
   results = model.predict(
       'path/to/image.jpg',
       conf={CONF_THRESH},
       iou={NMS_IOU_THRESH},
   )

   # Draw detections
   for r in results:
       img = r.plot()
       cv2.imshow('Weapon Detection', img)
       cv2.waitKey(0)

4. KEY PAPER HYPERPARAMETERS USED:
   - Optimizer:    SGD (momentum={MOMENTUM}, weight_decay={WEIGHT_DECAY})
   - LR:          Cosine annealing {LR0} → {LRF}
   - Batch:       {BATCH}, ImgSz: {IMGSZ}
   - NMS IoU:     {NMS_IOU_THRESH}, Confidence: {CONF_THRESH}
   - Epochs:      {EPOCHS} max, Early stopping patience={PATIENCE}

5. RESULT FILES:
   ablation_results.csv         → Ablation study comparison
   cross_validation_results.csv → 5-fold CV metrics per fold
   class_metrics.csv            → Per-class P/R/F1/AP on test set
   benchmark_results.csv        → Adverse condition robustness
"""

print(instructions)

# Save instructions to Drive
inst_path = f"{DRIVE_BASE}/INTEGRATION_INSTRUCTIONS.txt"
with open(inst_path, 'w') as f:
    f.write(instructions)
print(f"✅ Instructions saved: {inst_path}")

In [ ]:
# 7.5 — Final Summary Report

print("╔" + "═"*62 + "╗")
print("║         WEAPON DETECTION TRAINING — FINAL SUMMARY          ║")
print("╚" + "═"*62 + "╝")
print()

# Dataset
print("📊 DATASET:")
print(f"   Total images:    {len(all_pairs)}")
print(f"   Train:           {len(train_pairs)}")
print(f"   Val:             {len(val_pairs)}")
print(f"   Test:            {len(test_pairs)}")
print(f"   CV Folds:        {NUM_FOLDS}")

print()

# Ablation
print("🔬 ABLATION STUDY:")
if not df_ablation.empty and 'mAP@50' in df_ablation.columns:
    for _, row in df_ablation.iterrows():
        marker = ' ← BEST' if row['Model'] == BEST_MODEL_NAME else ''
        print(f"   {row['Model']:<10}: mAP@50={row['mAP@50']:.4f}, mAP@50:95={row['mAP@50:95']:.4f}{marker}")
else:
    print("   (Ablation not completed)")

print()

# Cross-validation
print(f"🔄 5-FOLD CROSS-VALIDATION ({BEST_MODEL_NAME.upper()}):")
if not df_cv.empty and 'mAP@50' in df_cv.columns:
    print(f"   mAP@50:     {df_cv['mAP@50'].mean():.4f} ± {df_cv['mAP@50'].std():.4f}")
    print(f"   mAP@50:95:  {df_cv['mAP@50:95'].mean():.4f} ± {df_cv['mAP@50:95'].std():.4f}")
    print(f"   F1:         {df_cv['F1'].mean():.4f} ± {df_cv['F1'].std():.4f}")
else:
    print("   (Cross-validation not completed)")

print()

# Per-class
print("🎯 PER-CLASS METRICS (Test Set — Normal):")
if not df_class.empty:
    for _, row in df_class.iterrows():
        print(f"   {row['Class']:<10}: AP@50={row['AP50']:.4f}, P={row['Precision']:.4f}, R={row['Recall']:.4f}, F1={row['F1']:.4f}")
else:
    print("   (Evaluation not completed)")

print()

# Drive location
print(f"💾 ALL RESULTS SAVED TO:")
print(f"   {DRIVE_BASE}")
print()
print("✅ TRAINING PIPELINE COMPLETE!")